# 피부질환 분류 — 모델 비교 실험

**목적**: 동일 데이터셋·동일 조건으로 5개 모델을 직접 학습해 EfficientNet-B4 선택 근거를 실험으로 증명  
**비교 모델**: ResNet-50 / MobileNetV3-L / DenseNet-121 / EfficientNet-B0 / EfficientNet-B4  
**출력물**: 비교 표(CSV) + 정확도 막대그래프 + 학습시간 그래프 → PPT P.71 삽입용

| 설정 | 값 |
|------|----|
| QUICK_COMPARE = True | 모델당 **5 epoch** (빠른 비교, ~30분) |
| QUICK_COMPARE = False | 모델당 **15 epoch** (정밀 비교, ~2시간) |

In [ ]:
!pip install -q timm

In [ ]:
# ── 한글 폰트 ────────────────────────────────────────
import subprocess, matplotlib
subprocess.run(['apt-get', 'install', '-y', '-q', 'fonts-nanum'], check=True)
font_dir = [p for p in matplotlib.font_manager.findSystemFonts() if 'Nanum' in p]
if font_dir:
    matplotlib.font_manager.fontManager.addfont(font_dir[0])
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
print('한글 폰트 설정 완료')

In [ ]:
import os, glob, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, f1_score, classification_report
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── 공통 설정 ─────────────────────────────────────────
BASE          = '/kaggle/input/datasets/kimminsu0109/skin-dataset'
TRAIN_DIR     = f'{BASE}/Train'
VAL_DIR       = f'{BASE}/Val'
OUTPUT_DIR    = '/kaggle/working'

BATCH_SIZE    = 16
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
LABEL_SMOOTH  = 0.1
USE_AMP       = True

# True: 모델당 5 epoch (빠른 비교)
# False: 모델당 15 epoch (정밀 비교)
QUICK_COMPARE = True
EPOCHS        = 5 if QUICK_COMPARE else 15
PATIENCE      = 3 if QUICK_COMPARE else 5

print(f'비교 모드: {"빠른" if QUICK_COMPARE else "정밀"} | 모델당 {EPOCHS} epoch')

In [ ]:
# ── 클래스 매핑 ───────────────────────────────────────
PREFIX_TO_KO = {
    'TS_광선각화증': '광선각화증', 'VS_광선각화증': '광선각화증',
    'TS_기저세포암': '기저세포암', 'VS_기저세포암': '기저세포암',
    'TS_멜라닌세포모반': '멜라닌세포모반', 'VS_멜라닌세포모반': '멜라닌세포모반',
    'TS_보웬병': '보웬병', 'VS_보웬병': '보웬병',
    'TS_비립종': '비립종', 'VS_비립종': '비립종',
    'TS_사마귀': '사마귀', 'VS_사마귀': '사마귀',
    'TS_악성흑색종': '악성흑색종', 'VS_악성흑색종': '악성흑색종',
    'TS_지루각화증': '지루각화증', 'VS_지루각화증': '지루각화증',
    'TS_편평세포암': '편평세포암', 'VS_편평세포암': '편평세포암',
    'TS_표피낭종': '표피낭종', 'VS_표피낭종': '표피낭종',
    'TS_피부섬유종': '피부섬유종', 'VS_피부섬유종': '피부섬유종',
    'TS_피지샘증식증': '피지샘증식증', 'VS_피지샘증식증': '피지샘증식증',
    'TS_혈관종': '혈관종', 'VS_혈관종': '혈관종',
    'TS_화농 육아종': '화농 육아종', 'VS_화농 육아종': '화농 육아종',
    'TS_흑색점': '흑색점', 'VS_흑색점': '흑색점',
}
KO_CLASSES   = sorted(set(PREFIX_TO_KO.values()))
KO_TO_IDX    = {ko: i for i, ko in enumerate(KO_CLASSES)}
CLASS_TO_IDX = {folder: KO_TO_IDX[ko] for folder, ko in PREFIX_TO_KO.items()}
IDX_TO_KO    = {i: ko for ko, i in KO_TO_IDX.items()}
NUM_CLASSES  = len(KO_CLASSES)
class_names  = [IDX_TO_KO[i] for i in range(NUM_CLASSES)]
print(f'클래스 수: {NUM_CLASSES}')

In [ ]:
# ── Dataset 클래스 ────────────────────────────────────
IMG_EXTS = ('*.png', '*.jpg', '*.jpeg')

class SkinDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, transform=None):
        self.transform = transform
        self.samples   = []
        for folder in sorted(os.listdir(img_dir)):
            if folder not in CLASS_TO_IDX:
                continue
            cls_idx    = CLASS_TO_IDX[folder]
            img_folder = os.path.join(img_dir, folder)
            paths = []
            for ext in IMG_EXTS:
                paths.extend(glob.glob(os.path.join(img_folder, ext)))
            for p in sorted(paths):
                self.samples.append((p, cls_idx))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, cls_idx = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, cls_idx


def make_loaders(input_size: int):
    """input_size에 맞는 DataLoader 쌍 반환. 매 모델마다 동일 split 사용."""
    train_tf = transforms.Compose([
        transforms.Resize((input_size + 20, input_size + 20)),
        transforms.RandomCrop(input_size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    train_ds   = SkinDataset(TRAIN_DIR, transform=train_tf)
    full_val   = SkinDataset(VAL_DIR,   transform=val_tf)
    val_size   = len(full_val) // 2
    test_size  = len(full_val) - val_size
    val_ds, test_ds = random_split(
        full_val, [val_size, test_size],
        generator=torch.Generator().manual_seed(SEED)
    )
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader

print('Dataset 클래스 정의 완료')

In [ ]:
# ── 모델 레지스트리 ───────────────────────────────────
# (이름, 빌더함수, 입력크기, 파라미터수(참고용))
def build_resnet50(n):
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, n)
    return m

def build_mobilenet(n):
    m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n)
    return m

def build_densenet121(n):
    m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    m.classifier = nn.Linear(m.classifier.in_features, n)
    return m

def build_efficientnet_b0(n):
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
    return m

def build_efficientnet_b4(n):
    m = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
    return m

MODEL_REGISTRY = [
    # (표시명,          빌더,                  입력크기, 파라미터수)
    ('ResNet-50',       build_resnet50,         224,     '25M'),
    ('MobileNetV3-L',   build_mobilenet,        224,     '5.4M'),
    ('DenseNet-121',    build_densenet121,      224,     '8M'),
    ('EfficientNet-B0', build_efficientnet_b0,  224,     '5.3M'),
    ('EfficientNet-B4', build_efficientnet_b4,  380,     '19M'),   # 최종 선택 모델
]

print('모델 레지스트리:')
for name, _, size, params in MODEL_REGISTRY:
    print(f'  {name:20s}  입력: {size}×{size}  파라미터: {params}')

In [ ]:
# ── 공통 학습/평가 함수 ───────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out  = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out  = model(imgs)
            loss = criterion(out, labels)
        total_loss += loss.item() * imgs.size(0)
        preds = out.argmax(1)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels


def run_experiment(model_name, builder, input_size):
    """단일 모델 학습 + Test 평가 → 결과 딕셔너리 반환."""
    print(f'\n{'='*60}')
    print(f'[{model_name}]  입력 {input_size}×{input_size}  |  {EPOCHS} epoch')
    print('='*60)

    train_loader, val_loader, test_loader = make_loaders(input_size)
    model     = builder(NUM_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_val_acc = 0.0
    no_improve   = 0
    best_state   = None
    history      = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    t0           = time.time()

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)

        print(f'  Epoch {epoch:02d}/{EPOCHS}  '
              f'Train {tr_acc:.4f}  Val {vl_acc:.4f}')

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve   = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break

    train_time = time.time() - t0

    # Best 가중치로 Test 평가
    model.load_state_dict(best_state)
    _, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion)
    f1  = f1_score(test_labels, test_preds, average='macro')
    f1w = f1_score(test_labels, test_preds, average='weighted')

    params = sum(p.numel() for p in model.parameters()) / 1e6

    print(f'  => Test Accuracy: {test_acc:.4f}  Macro-F1: {f1:.4f}  '
          f'학습시간: {train_time/60:.1f}분')

    return {
        'model':        model_name,
        'input_size':   input_size,
        'params_M':     round(params, 1),
        'test_acc':     round(test_acc, 4),
        'macro_f1':     round(f1, 4),
        'weighted_f1':  round(f1w, 4),
        'best_val_acc': round(best_val_acc, 4),
        'train_min':    round(train_time / 60, 1),
        'history':      history,
        'test_preds':   test_preds,
        'test_labels':  test_labels,
    }

print('함수 정의 완료')

In [ ]:
# ── 전체 비교 실험 실행 ───────────────────────────────
# ⚠️ GPU 메모리 부족 시 BATCH_SIZE = 8 로 줄이세요
results = []
for model_name, builder, input_size, _ in MODEL_REGISTRY:
    res = run_experiment(model_name, builder, input_size)
    results.append(res)
    torch.cuda.empty_cache()  # 모델 간 GPU 메모리 해제

print('\n\n모든 실험 완료!')

In [ ]:
# ── 비교 결과 표 ──────────────────────────────────────
df = pd.DataFrame([{
    '모델':          r['model'],
    '파라미터(M)':   r['params_M'],
    '입력 크기':     f"{r['input_size']}×{r['input_size']}",
    'Test Accuracy': r['test_acc'],
    'Macro F1':      r['macro_f1'],
    'Weighted F1':   r['weighted_f1'],
    'Best Val Acc':  r['best_val_acc'],
    '학습 시간(분)': r['train_min'],
} for r in results])

# EfficientNet-B4 행 강조
df['비고'] = df['모델'].apply(lambda x: '★ 최종 선택' if x == 'EfficientNet-B4' else '')
df_sorted  = df.sort_values('Test Accuracy', ascending=False).reset_index(drop=True)

print('=== 피부질환 분류 모델 비교 결과 ===')
print(df_sorted.to_string(index=False))

# CSV 저장
df_sorted.to_csv(os.path.join(OUTPUT_DIR, 'model_comparison.csv'), index=False, encoding='utf-8-sig')
print('\n✅ model_comparison.csv 저장 완료')

In [ ]:
# ── 시각화 1: Test Accuracy + Macro F1 비교 막대그래프 ─
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
model_names = [r['model'] for r in results]
test_accs   = [r['test_acc'] for r in results]
macro_f1s   = [r['macro_f1'] for r in results]

colors = ['#e74c3c' if n == 'EfficientNet-B4' else '#2980b9' for n in model_names]

# Accuracy
bars = axes[0].bar(model_names, test_accs, color=colors, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, test_accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_title('Test Accuracy 비교 (동일 조건)', fontsize=13)
axes[0].set_ylim(0, min(1.0, max(test_accs) + 0.08))
axes[0].tick_params(axis='x', rotation=25)
axes[0].set_ylabel('Accuracy')
axes[0].axhline(y=max(test_accs), color='red', linestyle='--', alpha=0.4)

# Macro F1
bars2 = axes[1].bar(model_names, macro_f1s, color=colors, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars2, macro_f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_title('Macro F1-Score 비교 (동일 조건)', fontsize=13)
axes[1].set_ylim(0, min(1.0, max(macro_f1s) + 0.08))
axes[1].tick_params(axis='x', rotation=25)
axes[1].set_ylabel('Macro F1')
axes[1].axhline(y=max(macro_f1s), color='red', linestyle='--', alpha=0.4)

# 범례
from matplotlib.patches import Patch
legend = [Patch(color='#e74c3c', label='EfficientNet-B4 (최종 선택)'),
          Patch(color='#2980b9', label='비교 모델')]
fig.legend(handles=legend, loc='upper center', ncol=2, fontsize=10, bbox_to_anchor=(0.5, 1.02))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison_accuracy.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ model_comparison_accuracy.png 저장')

In [ ]:
# ── 시각화 2: 파라미터 수 vs Test Accuracy 산점도 ──────
fig, ax = plt.subplots(figsize=(8, 6))

params_list = [r['params_M'] for r in results]
for i, r in enumerate(results):
    color  = '#e74c3c' if r['model'] == 'EfficientNet-B4' else '#2980b9'
    marker = '*' if r['model'] == 'EfficientNet-B4' else 'o'
    size   = 300 if r['model'] == 'EfficientNet-B4' else 120
    ax.scatter(r['params_M'], r['test_acc'], c=color, s=size, marker=marker,
               zorder=5, edgecolors='white', linewidths=1)
    ax.annotate(r['model'],
                xy=(r['params_M'], r['test_acc']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=9,
                color='#c0392b' if r['model'] == 'EfficientNet-B4' else '#2c3e50')

ax.set_xlabel('파라미터 수 (M)', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('파라미터 수 vs Test Accuracy\n(오른쪽 위 = 효율적)', fontsize=13)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'params_vs_accuracy.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ params_vs_accuracy.png 저장')

In [ ]:
# ── 시각화 3: 학습 곡선 비교 (Val Accuracy) ───────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_palette = ['#e74c3c', '#2980b9', '#27ae60', '#8e44ad', '#e67e22']

for i, r in enumerate(results):
    lw    = 2.5 if r['model'] == 'EfficientNet-B4' else 1.5
    alpha = 1.0 if r['model'] == 'EfficientNet-B4' else 0.75
    c     = colors_palette[i % len(colors_palette)]
    ep    = range(1, len(r['history']['val_acc']) + 1)

    axes[0].plot(ep, r['history']['val_loss'], label=r['model'],
                 color=c, linewidth=lw, alpha=alpha)
    axes[1].plot(ep, r['history']['val_acc'],  label=r['model'],
                 color=c, linewidth=lw, alpha=alpha)

axes[0].set_title('Val Loss 비교', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=9)

axes[1].set_title('Val Accuracy 비교', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison_curves.png'), dpi=150)
plt.show()
print('✅ model_comparison_curves.png 저장')

In [ ]:
# ── 시각화 4: PPT용 최종 비교 표 이미지 ─────────────────
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')

table_data = [
    [r['model'], f"{r['params_M']}M", f"{r['input_size']}×{r['input_size']}",
     f"{r['test_acc']*100:.1f}%", f"{r['macro_f1']:.3f}",
     f"{r['train_min']}분",
     '★ 최종 선택' if r['model'] == 'EfficientNet-B4' else '']
    for r in sorted(results, key=lambda x: x['test_acc'], reverse=True)
]

col_labels = ['모델', '파라미터', '입력 크기', 'Test Acc', 'Macro F1', '학습 시간', '비고']
tbl = ax.table(cellText=table_data, colLabels=col_labels,
               loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1, 2)

# EfficientNet-B4 행 강조 (빨간 배경)
for row_idx, row in enumerate(table_data):
    if row[0] == 'EfficientNet-B4':
        for col_idx in range(len(col_labels)):
            tbl[row_idx + 1, col_idx].set_facecolor('#fdecea')
            tbl[row_idx + 1, col_idx].set_text_props(fontweight='bold', color='#c0392b')

# 헤더 색상
for col_idx in range(len(col_labels)):
    tbl[0, col_idx].set_facecolor('#1e2d5a')
    tbl[0, col_idx].set_text_props(color='white', fontweight='bold')

plt.title('피부질환 분류 모델 비교 결과 (동일 데이터셋 / 동일 조건 학습)',
          fontsize=13, pad=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison_table.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ model_comparison_table.png 저장 — PPT에 바로 삽입 가능')

In [ ]:
# ── 최종 요약 출력 ────────────────────────────────────
best = max(results, key=lambda x: x['test_acc'])
print('\n' + '='*60)
print('최종 비교 요약')
print('='*60)
print(f"  최고 성능 모델 : {best['model']}")
print(f"  Test Accuracy  : {best['test_acc']:.4f}")
print(f"  Macro F1       : {best['macro_f1']:.4f}")
print(f"  학습 시간      : {best['train_min']}분")
print()
print('생성된 파일:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    if any(f.endswith(ext) for ext in ['.png', '.csv']):
        size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
        print(f'  {f:45s}  {size:.0f} KB')